<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/Mlondie_ask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Question 2 - Part 1: Dataset Creation

Create a DataFrame with 15 participant IDs and their initial randomly assigned fitness levels (Low : 0, Moderate: 1, or High: 2).


In [ ]:
import warnings
import pandas as pd
import numpy as np

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:

participant_ids = [f'Participant_{i+1}' for i in range(15)]
initial_fitness_levels = np.random.randint(0, 3, 15)

df = pd.DataFrame({
    'Participant_ID': participant_ids,
    'Initial_Fitness_Level': initial_fitness_levels
})

df_1 = df.copy()

In [ ]:
import numpy as np
df_2 = df.copy()

# Define parameters for each fitness level
fitness_params = {
    0: {'mean': 6000, 'std_dev': 600},
    1: {'mean': 7500, 'std_dev': 500},
    2: {'mean': 9000, 'std_dev': 700}
}

# Generate daily step counts for 14 days
for day in range(1, 15):
    column_name = f'Steps Day {day}'

    # Create an empty list to store steps for each person
    steps = []

    # Loop through each row in df
    for level in df['Initial_Fitness_Level']:
        mean = fitness_params[level]['mean']
        std_dev = fitness_params[level]['std_dev']
        value = np.random.normal(mean, std_dev)
        value = round(value)
        value = min(max(value, 3000), 15000)  # clip between 3000 and 15000
        steps.append(int(value))

    # Add column to DataFrame
    df_2[column_name] = steps

# Display the updated DataFrame with step counts
steps_cols = [col for col in df_2.columns if 'Part' in col]+[col for col in df_2.columns if 'Steps' in col]


In [ ]:
# Select the columns containing step counts
step_columns = [f'Steps Day {day}' for day in range(1, 15)]
steps_array = df_2[step_columns].values


In [ ]:
# fitness assigned to each participant
display(df_1)

In [ ]:
display(df_2[steps_cols])

## Question 2 - Part 2: Data Analysis

In [ ]:
# Calculate the average daily steps for each participant
df_2['Average_Daily_Steps'] = df_2[step_columns].mean(axis=1)

# Sort the participants by average daily steps in descending order
df_sorted = df_2.sort_values(by='Average_Daily_Steps', ascending=False)

# Display the top 5 participants and their average daily steps
display(df_sorted[['Participant_ID', 'Average_Daily_Steps']].head())

In [ ]:
# Overall mean and standard deviation of all step counts
overall_mean_steps = round(steps_array.mean())
overall_std_steps = round(steps_array.std())

# Display overall mean and standard deviation
print(f'Overall Mean Steps: {overall_mean_steps}')
print(f'Overall Standard Deviation Steps: {overall_std_steps}')

In [ ]:
# Median daily steps for each participant
df_2['Median_Daily_Steps'] = df_2[step_columns].median(axis=1)

# Participants with the highest median daily steps
highest_median_steps = df_2['Median_Daily_Steps'].max()
participants_highest_median = df_2[df_2['Median_Daily_Steps'] == highest_median_steps][['Participant_ID', 'Median_Daily_Steps']]

# Participant with the lowest median daily steps
lowest_median_steps = df_2['Median_Daily_Steps'].min()
participants_lowest_median = df_2[df_2['Median_Daily_Steps'] == lowest_median_steps][['Participant_ID', 'Median_Daily_Steps']]

# Display the participant with the highest median daily steps
print("Participant(s) with the Highest Median Daily Steps:")
display(participants_highest_median)

# Display the participant with the lowest median daily steps
print("\nParticipant(s) with the Lowest Median Daily Steps:")
display(participants_lowest_median)

In [ ]:
# Count participants with average daily steps above 8000
participants_above_8000 = df_sorted[df_sorted['Average_Daily_Steps'] > 8000]
count_above_8000 = len(participants_above_8000)

# Display the count
print(f'Number of participants with average daily steps above 8000: {count_above_8000}')

In [ ]:
# Compute the percentiles
percentiles = np.percentile(steps_array, [25, 50, 75])

# Display the percentiles
print(f"25th Percentile: {round(percentiles[0])}")
print(f"50th Percentile (Median): {round(percentiles[1])}")
print(f"75th Percentile: {round(percentiles[2])}")

## Question 3 - Part 1: Dataset Preparation and Cleaning

In [ ]:
# Load data from Github
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/adult.csv"
input_data = pd.read_csv(url)
input_data.head()

In [ ]:
input_data.info()

In [ ]:
# Replace . with _ column names
input_data.columns = input_data.columns.str.replace('.', '_', regex=False)

In [ ]:
# Columns with ? as missing values and their count
(input_data == "?").sum()

In [ ]:
input_data.head()

In [ ]:
input_data.replace("?", np.nan, inplace=True)

In [ ]:
df = input_data.copy()

In [ ]:
# Separate columns by data type
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = [np.number]).columns

In [ ]:
# Separate columns by data type
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = [np.number]).columns

# Impute categorical columns with mode (most frequent value)
for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col].fillna(mode_value, inplace = True)

# Impute numerical columns with median
for col in numerical_cols:
    median_value = df[col].median()
    df[col].fillna(median_value, inplace = True)

df.drop_duplicates(inplace = True)

In [ ]:
# check that no missing values remain
print(df.isnull().sum())

## Question 3 -  Part 2: Data Visualisation

In [ ]:
import matplotlib.pyplot as plt

# Create figure and subplots
fig, axes = plt.subplots(2, 2, figsize = (14, 10))
plt.subplots_adjust(hspace=0.4, wspace = 0.3)

#Top-Left: Stacked Bar Chart

ax1 = axes[0, 0]

gender_income = df.groupby(['sex', 'income']).size().unstack(fill_value = 0)
gender_income.plot(kind = 'bar', stacked = True, ax = ax1,
                   color = ['#72B7B2', '#E26D5A'])

ax1.set_title('Distribution by Gender and Income', fontsize=12, fontweight='bold')
ax1.set_xlabel('Gender')
ax1.set_ylabel('Number of Individuals')
ax1.legend(title = 'Income Category')
ax1.grid(axis = 'y', linestyle = '--', alpha = 0.6)

# Annotation
for p in ax1.patches:
    if p.get_height() > 0:
        ax1.annotate(int(p.get_height()),
                     (p.get_x() + p.get_width() / 2, p.get_y() + p.get_height() / 2),
                     ha='center', va='center', fontsize=9, color='white')

# Top-Right: Line Graph

ax2 = axes[0, 1]

age_hours = df.groupby(['age', 'income'])['hours_per_week'].mean().unstack()
age_hours.plot(ax = ax2, linewidth = 2)

ax2.set_title('Average Hours Worked per Week by Age', fontsize = 12, fontweight = 'bold')
ax2.set_xlabel('Age (years)')
ax2.set_ylabel('Average Hours per Week')
ax2.legend(title = 'Income Category')
ax2.grid(True, linestyle = '--', alpha = 0.6)

# Highlight
peak_age = age_hours['>50K'].idxmax()
peak_hours = age_hours['>50K'].max()
ax2.annotate(f'Peak: {peak_hours:.1f} hrs', xy=(peak_age, peak_hours),
             xytext=(peak_age + 5, peak_hours + 2),
             arrowprops=dict(facecolor='black', arrowstyle='->'), fontsize=9)

# Bottom-Left: Histogram

ax3 = axes[1, 0]

df_low = df[df['income'] == '<=50K']['hours_per_week']
df_high = df[df['income'] == '>50K']['hours_per_week']

ax3.hist([df_low, df_high],
         bins = 20, color = ['#72B7B2', '#E26D5A'], label = ['<=50K', '>50K'],
         alpha = 0.8, edgecolor = 'black')

ax3.set_title('Distribution of Weekly Work Hours by Income Group', fontsize=12, fontweight='bold')
ax3.set_xlabel('Hours per Week')
ax3.set_ylabel('Frequency')
ax3.legend(title = 'Income Group')
ax3.grid(axis = 'y', linestyle = '--', alpha = 0.6)

# Bottom-Right: Grouped Bar Chart
ax4 = axes[1, 1]

edu_occ = df.groupby(['occupation', 'income'])['education_num'].mean().unstack()
edu_occ.plot(kind = 'bar', ax = ax4, width = 0.8, color = ['#72B7B2', '#E26D5A'])

ax4.set_title('Average Education Level by Occupation and Income', fontsize = 12, fontweight = 'bold')
ax4.set_xlabel('Occupation')
ax4.set_ylabel('Average Education Level (years)')
ax4.legend(title = 'Income Group')
ax4.tick_params(axis = 'x', rotation = 45)
ax4.grid(axis = 'y', linestyle = '--', alpha = 0.6)

# Annotate outlier
#max_edu = edu_occ['>50K'].max()
#max_occ = edu_occ['>50K'].idxmax()
#ax4.annotate('Highest avg. education',
#             xy=(list(edu_occ.index).index(max_occ), max_edu),
#             xytext=(1, max_edu + 0.5),
#             arrowprops=dict(facecolor='black', arrowstyle='->'),
#             fontsize=9)

# --------------------------------------------
# Apply overall styling and show figure
# --------------------------------------------
fig.suptitle('Demographic and Income Analysis Dashboard', fontsize=15, fontweight = 'bold')
plt.show()

